# Check geographical spread of patents and publications

In [41]:
import pandas as pd
import json
import altair as alt

from discovery_child_development import PROJECT_DIR, S3_BUCKET
from discovery_child_development.utils import plotting_utils as pu


ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

In [73]:
from discovery_child_development.utils.jsonl_utils import load_jsonl

In [2]:
relevant_df = pd.read_csv(ENRICHED_DATA_DIR / 'relevant_labelled_df.csv')

In [5]:
relevant_df.head(1)

,id,text,Dataset,topics,Detection,Detection_,Management,Management_,year,country_code
0,CN-107945066-A,Kindergarten intelligent control system and co...,Patents,robotics,0.625067,1,0.568404,1,2018,CN


In [20]:
patents_totals_df = (
    pd.DataFrame(json.load(open("total_patents_per_country.json", "r")))
    .astype({"publication_year": int, "total_publications": int})
    .rename(columns={"publication_year": "year"})
    .query("year >= 2013 and year < 2024")
)

## Patents

In [24]:
patents_totals_df.head(1)

,year,country_code,total_publications
0,2013,AP,1094


In [29]:
counts_df = (
    relevant_df
    .query("Dataset == 'Patents'")
    .query("year >= 2013 and year < 2024")
    .groupby('country_code')
    .agg(counts=('id', 'count'))
    .reset_index()
    .assign(fraction = lambda df: df['counts'] / df['counts'].sum())
)

counts_totals_df = (
    patents_totals_df
    .groupby("country_code")
    .agg(counts=('total_publications', 'sum'))
    .reset_index()
    .assign(fraction = lambda df: df['counts'] / df['counts'].sum())    
)    


In [38]:
patent_comparison = (
    counts_df
    .merge(counts_totals_df, on='country_code', suffixes=('_patents', '_totals'))
    .assign(fraction_diff = lambda df: df['fraction_patents']/df['fraction_totals'])
    .sort_values('counts_patents', ascending=False)
)

In [172]:
patent_comparison.to_csv("global_vs_data_patents.csv", index=False)

## Publications

In [43]:
(
    relevant_df
    .query("Dataset == 'Publications'")
)

,id,text,Dataset,topics,Detection,Detection_,Management,Management_,year,country_code
11534,W2146349261,The impact of pretend play on children's devel...,Publications,"arts, communication, cognitive, games, preschool",0.198959,0,0.934951,1,2013,NaN
11535,W2134744704,The Effects of Poverty on Childhood Brain Deve...,Publications,"income, neuroscience, inequality, mental_healt...",0.905183,1,0.444455,0,2013,NaN
11536,W2124222455,Investing in Preschool Programs. We summarize ...,Publications,"income, cognitive, social_services",0.175922,0,0.888656,1,2013,NaN
11537,W2022384642,Evidence for General and Domain‐Specific Eleme...,Publications,"communication, cognitive, operations",0.662059,1,0.739743,1,2013,NaN
11538,W2055043956,The role of nutrition in children's neurocogni...,Publications,"cognitive, nutrition",0.822885,1,0.677591,1,2013,NaN
...,...,...,...,...,...,...,...,...,...,...
49800,W4380048305,Trilingual families' language strategies: pote...,Publications,communication,0.401767,0,0.797585,1,2023,NaN
49801,W4385650553,The Effects of Vitamin D Supplementation on Re...,Publications,"rct, nutrition, health",0.992909,1,0.000782,0,2023,NaN
49802,W4367694030,A Study to Assess the Effectiveness of Video A...,Publications,"social_services, protection",0.002119,0,0.996927,1,2023,NaN
49803,W4380886404,Vaginal Bleeding In Prepubertal Girls-A Case S...,Publications,NaN,0.576360,1,0.154030,0,2023,NaN


In [66]:
# Use requests to call OpenAlex API
import requests

url = "https://api.openalex.org/works?filter=openalex_id:W2146349261|W2134744704"
response = requests.get(url)

In [134]:
openalex_metadata = load_jsonl("publications_metadata.jsonl")
already_fetched_ids = [p['id'].split("/")[-1] for p in openalex_metadata]
print(f"Already fetched {len(already_fetched_ids)} publications")

Already fetched 38271 publications


In [133]:
# create batches
batch_size = 50

ids = relevant_df.query("Dataset == 'Publications'")['id'].tolist()
ids = [i for i in ids if i not in already_fetched_ids]

batches = [ids[i:i + batch_size] for i in range(0, len(ids), batch_size)]

print(f"Number of documents left: {len(ids)}")

with open("publications_metadata.jsonl", "a") as f:
    for i, batch in enumerate(batches):
        url = f"https://api.openalex.org/works?filter=openalex_id:{'|'.join(batch)}&per-page=50"
        response = requests.get(url)
        results_json = response.json()['results']
        # write each dict to a new line
        for result in results_json:
            f.write(json.dumps(result) + "\n")
        print(f"{i+1}/{len(batches)} batches processed")

Number of documents left: 38121
1/763 batches processed
2/763 batches processed
3/763 batches processed
4/763 batches processed
5/763 batches processed
6/763 batches processed
7/763 batches processed
8/763 batches processed
9/763 batches processed
10/763 batches processed
11/763 batches processed
12/763 batches processed
13/763 batches processed
14/763 batches processed
15/763 batches processed
16/763 batches processed
17/763 batches processed
18/763 batches processed
19/763 batches processed
20/763 batches processed
21/763 batches processed
22/763 batches processed
23/763 batches processed
24/763 batches processed
25/763 batches processed
26/763 batches processed
27/763 batches processed
28/763 batches processed
29/763 batches processed
30/763 batches processed
31/763 batches processed
32/763 batches processed
33/763 batches processed
34/763 batches processed
35/763 batches processed
36/763 batches processed
37/763 batches processed
38/763 batches processed
39/763 batches processed
40

## Get reference data

In [165]:
all_country_codes = [
    'AD',
    'AE',
    'AF',
    'AL',
    'AM',
    'AO',
    'AR',
    'AS',
    'AT',
    'AU',
    'AZ',
    'BA',
    'BB',
    'BD',
    'BE',
    'BF',
    'BG',
    'BH',
    'BI',
    'BJ',
    'BN',
    'BO',
    'BR',
    'BS',
    'BT',
    'BW',
    'BY',
    'CA',
    'CD',
    'CG',
    'CH',
    'CI',
    'CL',
    'CM',
    'CN',
    'CO',
    'CR',
    'CU',
    'CV',
    'CW',
    'CY',
    'CZ',
    'DE',
    'DJ',
    'DK',
    'DM',
    'DO',
    'DZ',
    'EC',
    'EE',
    'EG',
    'ES',
    'ET',
    'FI',
    'FJ',
    'FO',
    'FR',
    'GA',
    'GB',
    'GD',
    'GE',
    'GH',
    'GL',
    'GM',
    'GP',
    'GR',
    'GT',
    'GU',
    'GW',
    'GY',
    'HK',
    'HN',
    'HR',
    'HT',
    'HU',
    'ID',
    'IE',
    'IL',
    'IN',
    'IQ',
    'IR',
    'IS',
    'IT',
    'JM',
    'JO',
    'JP',
    'KE',
    'KG',
    'KH',
    'KI',
    'KR',
    'KW',
    'KZ',
    'LA',
    'LB',
    'LC',
    'LK',
    'LR',
    'LT',
    'LU',
    'LV',
    'LY',
    'MA',
    'MD',
    'MG',
    'MK',
    'ML',
    'MM',
    'MN',
    'MO',
    'MP',
    'MQ',
    'MR',
    'MT',
    'MU',
    'MW',
    'MX',
    'MY',
    'MZ',
    'NA',
    'NC',
    'NE',
    'NG',
    'NI',
    'NL',
    'NO',
    'NP',
    'NZ',
    'OM',
    'PA',
    'PE',
    'PG',
    'PH',
    'PK',
    'PL',
    'PR',
    'PS',
    'PT',
    'PY',
    'QA',
    'RO',
    'RS',
    'RU',
    'RW',
    'SA',
    'SC',
    'SD',
    'SE',
    'SG',
    'SI',
    'SK',
    'SL',
    'SN',
    'SR',
    'SS',
    'SV',
    'SZ',
    'TG',
    'TH',
    'TJ',
    'TL',
    'TN',
    'TR',
    'TT',
    'TW',
    'TZ',
    'UA',
    'UG',
    'US',
    'UY',
    'UZ',
    'VC',
    'VE',
    'VN',
    'WS',
    'XK',
    'YE',
    'ZA',
    'ZM',
    'ZW'
]

In [169]:
years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
years = [str(y) for y in years]

counts = []

for i, country_code in enumerate(all_country_codes):
    url = f"https://api.openalex.org/works?filter=institutions.country_code:{country_code},publication_year:{'|'.join(years)}&per-page=1"
    response = requests.get(url)
    counts.append(response.json()['meta']['count'])
    print(f"Processed {i}/{len(all_country_codes)}")


Processed 0/180
Processed 1/180
Processed 2/180
Processed 3/180
Processed 4/180
Processed 5/180
Processed 6/180
Processed 7/180
Processed 8/180
Processed 9/180
Processed 10/180
Processed 11/180
Processed 12/180
Processed 13/180
Processed 14/180
Processed 15/180
Processed 16/180
Processed 17/180
Processed 18/180
Processed 19/180
Processed 20/180
Processed 21/180
Processed 22/180
Processed 23/180
Processed 24/180
Processed 25/180
Processed 26/180
Processed 27/180
Processed 28/180
Processed 29/180
Processed 30/180
Processed 31/180
Processed 32/180
Processed 33/180
Processed 34/180
Processed 35/180
Processed 36/180
Processed 37/180
Processed 38/180
Processed 39/180
Processed 40/180
Processed 41/180
Processed 42/180
Processed 43/180
Processed 44/180
Processed 45/180
Processed 46/180
Processed 47/180
Processed 48/180
Processed 49/180
Processed 50/180
Processed 51/180
Processed 52/180
Processed 53/180
Processed 54/180
Processed 55/180
Processed 56/180
Processed 57/180
Processed 58/180
Process

In [170]:
pd.DataFrame({
    "country_code": all_country_codes,
    "total_publications": counts
}).to_json("total_publications_per_country.json", orient="records")

In [171]:
pd.DataFrame({
    "country_code": all_country_codes,
    "total_publications": counts
})


,country_code,total_publications
0,AD,385
1,AE,104268
2,AF,5359
3,AL,13335
4,AM,17560
...,...,...
175,XK,8636
176,YE,17385
177,ZA,363382
178,ZM,14568
